# Create Training for Content Based Filtering Assignment

Once the data has been trimmed in TrimDataset.ipynb, create the user and data features and write out training and test vectors.
The output is
- content_movie_list.csv   movie id, movie title
- content_item_vecs.csv  item training vectors for all movies, one vector per genre
- content_item_train.csv  movie training vectors
- content_item_train_header.txt  lists the headers in item_train.csv  (todo: just incorporate into item_train.csv)
- content_user_train.csv  user training vectors
- content_user_train_header.txt  lists the headers in user_train.csv  (todo: just incorporate into user_train.csv)
- content_y_train.csv  the rating for the training vectors

In [ ]:
import re
import csv
import pandas as pd
import numpy as np
from numpy.random import default_rng
from collections import defaultdict
min_rating_count = 10
genre_names = ['Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller']
num_genres = len(genre_names)

In [ ]:
movie_df  = pd.read_csv('post_trim_movie_df.csv',  index_col=0)
rating_df = pd.read_csv('post_trim_rating_df.csv', index_col=0)
print(movie_df.columns)
print(f"number of movies {len(movie_df.index)}")

In [ ]:
def add_ave_rating():
    """" adds a movie average rating into movie_df """
    if 'ave rating' in movie_df.columns:
        movie_df.drop('ave rating', axis=1, inplace=True)
    movie_df.insert(3,"ave rating",0)
    
    for index, row in movie_df.iterrows():
        #print(movie_df.loc[index,"title"])
        movie_id = row["movie id"]
        mid_df = rating_df.loc[rating_df["movie id"] == movie_id]
        if (len(mid_df.index) < min_rating_count):
            print(f"error? too few ratings {len(mid_df.index)}, movie_id = {movie_id}")
        #print(mid_df.columns)
        top = np.sum( mid_df["rating"].tolist())
        bot = len( mid_df["rating"].index )
        if bot == 0: print (f"error, {top}, {bot}")
        r_ave = np.divide(top, bot)
        movie_df.loc[index,"ave rating"] = r_ave

In [ ]:
add_ave_rating()
movie_df.columns

In [ ]:
def eng_user_features():
    """ ave rating / genre, average rating """
    user_to_genre = defaultdict()
    for i in range(len(rating_df.index)):
        user_id = rating_df.loc[i,"user id"]
        movie_id = rating_df.loc[i,"movie id"]
        rating  = rating_df.loc[i,"rating"]
        #print(user_id)
        if user_id not in user_to_genre:
            user_to_genre[user_id] = {}
            user_to_genre[user_id]["glist"] = np.zeros(num_genres)
            user_to_genre[user_id]["g_count"] = np.zeros(num_genres)
            user_to_genre[user_id]["rating_count"] = 0
            user_to_genre[user_id]["rating_sum"] = 0
            user_to_genre[user_id]["movies"] = {}  # rating per movie
 
        # ave rating/genres per user
        item_genres = movie_df.loc[movie_df["movie id"] == movie_id, genre_names[0]:genre_names[-1]].to_numpy()  # multi-hot
        #if user_id == 1: print(item_genres)
        user_to_genre[user_id]["glist"] = np.sum( (user_to_genre[user_id]["glist"], item_genres * rating), axis = 0)
        user_to_genre[user_id]["g_count"] = np.sum( (user_to_genre[user_id]["g_count"],  item_genres), axis = 0)

        # ave rating for a user                   
        user_to_genre[user_id]["rating_count"] += 1    
        user_to_genre[user_id]["rating_sum"] += rating  #accumulate for average

        # create a dictionary of movies rated by each user as long as we are at it. 
        user_to_genre[user_id]["movies"][movie_id] = rating

    # do averages
    #https://stackoverflow.com/questions/26248654/how-to-return-0-with-divide-by-zero divide hint.
    for user_id in user_to_genre.keys():
        #print(user_id)
        a = user_to_genre[user_id]["glist"]
        b = user_to_genre[user_id]["g_count"]
        user_to_genre[user_id]["glist"] = np.divide( a, b, out=np.zeros_like(a), where=b != 0)  #avoids div by zero for empty locations

        user_to_genre[user_id]["rating_ave"] = user_to_genre[user_id]["rating_sum"]/user_to_genre[user_id]["rating_count"]
    return(user_to_genre)

    

In [ ]:
def create_user_df():
    # add ave rating per genre to user_feature_df
    # add content to columns
    data = []
    for user_id in user_to_genre:
        for idx, name in enumerate(genre_names):
            glist = user_to_genre[user_id]["glist"]
        #print(glist[0])
        data.append([user_id, user_to_genre[user_id]["rating_count"], user_to_genre[user_id]["rating_ave"], *glist[0]] )
        #print(data)
    user_df = pd.DataFrame(data)
    user_df.columns = ["user id", "rating count", "rating ave"] + genre_names
    return(user_df)


In [ ]:
# debug command to display user data
def show_user(user_id):
    a = user_to_genre[user_id]
    print(f'glist        : {a["glist"]}')
    print(f'g_count      : {a["g_count"]}')
    print(f'rating ave   : {a["rating_ave"]}')
    print(f'rating count : {a["rating_count"]}')


In [ ]:
user_to_genre = eng_user_features()
user_df = create_user_df()
print(f"number of users: {len(user_df.index)}")
user_df.columns

In [ ]:
def get_genres(glist):
    locs = []
    for i, g in enumerate(glist):
        if g == 1:
            locs.append(i)
    return(locs)

def make_training(num_selected_users, num_genre, user_df, allusers=False, ):
    """ this version will make a training example with one genre per movie. 
    It replicates those that had multiple genres selected """
    subset = ['movie id', 'year', 'ave rating', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller']
    count = 0
    if allusers == False:
      selected_users = select_users(num_selected_users)
    else: 
      selected_users = user_df["user id"].tolist()  # all users
    
    with open('./parsed/content_user_train.csv', 'w') as user_f, \
         open('./parsed/content_item_train.csv', 'w') as item_f,\
         open('./parsed/content_y_train.csv', 'w') as y_f:

        for i in range(len(rating_df.index)):  # train example(s) per rating
            user_id  = rating_df.loc[i,"user id"]
            movie_id = rating_df.loc[i,"movie id"]
            rating   = rating_df.loc[i,"rating"]
            
            if user_id in selected_users:
                movie_idx = movie_df.loc[movie_df["movie id"] == movie_id].index.to_list()  #must be a better way
                if len(movie_idx) > 1: print("error, movie_idx should be 1")
                movie_idx = movie_idx[0]
                glist = movie_df.loc[movie_idx, genre_names[0]:genre_names[-1]].to_list()
                # split into a training example per genre
                for g in get_genres(glist):
                    head = movie_df.loc[movie_idx, ['movie id', 'year',  'ave rating'] ].to_list()
                    tmp = [0] * num_genre
                    tmp[g] = 1
                    movie_str = ','.join(str(x) for x in (head + tmp))
                    item_f.write(f'{movie_str}\n')

                    user_features = user_df.loc[user_df["user id"] == user_id].values 
                    user_features_str = ','.join(str(x) for x in user_features[0])
                    user_f.write(f'{user_features_str}\n')

                    y_f.write(f'{rating}\n')
                    count += 1
                    if count % 1000 == 0: print(f"{count}, ", end="")
            
    with open('./parsed/content_item_train_header.txt', 'w') as f:
        feature_str = ','.join(str(x) for x in subset)
        f.write(f'{feature_str}\n')

    with open('./parsed/content_user_train_header.txt', 'w') as f:
        feature_str = ','.join(str(x) for x in user_df.columns)
        f.write(f'{feature_str}\n')
    print(f"wrote {count} training examples")

def make_item_vecs(num_genre):
  """ writes out all the vectors for all the movies in item_features_df 
      - it replicates lines with multiple genres 
      - this allows the assigment to create a new user on the fly and make predictions
  """
  with open('./parsed/content_item_vecs.csv', 'w') as f:
    for movie_idx, row in movie_df.iterrows():
        glist = movie_df.loc[movie_idx, genre_names[0]:genre_names[-1]].to_list()
        for g in get_genres(glist):
            head = movie_df.loc[movie_idx, ['movie id', 'year',  'ave rating'] ].to_list()
            tmp = [0] * num_genre
            tmp[g] = 1
            movie_str = ','.join(str(x) for x in (head + tmp))
            f.write(f'{movie_str}\n')

def write_movie_list():
    """ writes a list of all movies in the db  """
    mlist = movie_df["movie id"].to_list()
    count = 0
    with open('./movies.csv', newline='') as infile, open('./parsed/content_movie_list.csv', 'w') as outfile:
        reader = csv.reader(infile, delimiter=',', quotechar='"')
        writer = csv.writer(outfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1  #skip header
                writer.writerow(line) 
            else:
                count +=1
                movie_id = int(line[0]) 
                if movie_id in mlist:
                    writer.writerow(line) 

#def make_movie_list():
#    """ writes a list of all movies in the db  """
#    with open('./parsed/movie_list.csv', 'w') as f:
#        for movie_idx, row in movie_df.iterrows():
#            mlist = movie_df.loc[movie_idx,  ["movie id", "title"]].to_list()
#            #print(mlist)
#            movie_str = f"{mlist[0]}, '{mlist[1]}'"
#            f.write(f'{movie_str}\n')


In [ ]:
all = len(user_df.index)
#user_t_df = user_df.drop('rating_ave',axis=1)  #keep user rating, drop it when using it.
make_training(all,num_genres, user_df,allusers=True)  

In [ ]:
make_item_vecs(num_genres)

In [ ]:
write_movie_list()

In [ ]:
# don't use these
#rating_df.to_csv('post_create_rating_df.csv', float_format="%.2f")
#movie_df.to_csv('post_create_movie_df.csv', float_format="%.2f")
#user_df.to_csv('post_create_user_df.csv', float_format="%.2f")

#do use this
import pickle
with open('./parsed/content_user_to_genre.pickle', 'wb') as f:
    pickle.dump(user_to_genre, f, protocol=pickle.HIGHEST_PROTOCOL)

